# Viewing structures

- **`pf.view()`**: one or more structures, with each chain's sequence linked to the 3D view. It's built into the package, so no extra install is needed.
- **`pf.py3Dmol()`**: a plain py3Dmol overlay that you can keep customizing. It needs `pip install 'pdbframe[view]'`.

Both load 3Dmol.js from the internet. This notebook is saved without outputs; run it to see the views, or open `3tfk_view.html` in a browser.

In [ ]:
from pdbframe import PDBFrame
import pdbframe

ref = PDBFrame.read('data/3tfk_rcsb.pdb')            # experimental structure
model = PDBFrame.read('data/3tfk_af2_renamed.pdb')   # AlphaFold2 model, chains named differently

# 1. `view()`: sequence linked to 3D

Press **?** in the viewer for the full list of controls. In short:

| | |
|---|---|
| **Sequences ▾** | which chains' sequences to show; none at first unless you pass `sequence=` |
| **Hover** | a residue lights up in both panels |
| **Drag** across the sequence | select a range; click a chain label to select the whole chain |
| **Ctrl-click** (⌘-click on Mac) | pick residues or chains; distances between picks are listed and drawn |
| **Rotate / move / zoom** | drag / Ctrl+drag or middle-button drag / scroll or Shift+drag |
| **3D chains ▾**, **Surface** | hide chains in 3D, show the molecular surface |
| **Reset view**, **Copy as code**, **Save PNG**, **Clear** | back to the start; the selection as a `pf.select(...)` call; an image; clear |

The arrows in the bottom-left corner show the x (red), y (green) and z (blue) directions as the view turns. Residue numbers above the sequence are the file's numbering. Chains have the same colors as in `py3Dmol()`.

In [ ]:
ref.view(sequence=True)   # sequence=['A', 'C'] for some chains; none shown by default

## Mark regions, and save the page
`regions` is `{name: residues}`. The residues can be a chain ID, an inclusive residue-number range like `('A', 50, 85)`, any selection such as `ref.near(...)`, or a list of these. `region_styles` also draws a region as sticks, and `interfaces=True` marks every chain-chain interface. The `.save()` call writes the page `3tfk_view.html` next to this notebook.

In [ ]:
peptide = ref.select(chains='C')
pocket = ref.near(peptide, cutoff=4).select(chains='C', invert=True)   # residues touching the peptide

v = ref.view(regions={'peptide pocket': pocket, 'alpha1 helix': ('A', 50, 85), 'peptide': 'C'},
             region_styles={'peptide pocket': 'stick'}, sequence=['A', 'C'])
v.save('3tfk_view.html')
v

## Compare a model with the reference, colored by lDDT
- **`compare=`** draws another structure (grey) in the same space. Residues are paired by sequence, so hovering one lights up its partner.
- **`superpose=['A']`** fits the model on the reference's chain A before drawing; `'all'` fits on all chains. Your frames are not moved.
- **`values=`** colors residues by a per-residue table, red (low) to blue (high). `values='plddt'` colors a predicted model by its own confidence.

In [ ]:
lddt = model.metrics.lddt(ref, per_residue=True)
ref.view(compare=model, superpose=['A'], values=lddt, sequence=['A', 'C'])

## Several structures
`pdbframe.view([...])` shows any number of structures, the first one leading. Sequences are stacked in aligned blocks, and **Structures ▾** / **One at a time** (◀ ▶ or the arrow keys) flip between them. Per-structure options take lists: `style`, `colors`, `values` and `regions`. `compact=True` keeps big ensembles small.

In [ ]:
moved = model.transform([[1, 0, 0], [0, 1, 0], [0, 0, 1]], [4, 0, 0])   # a stand-in second model
pdbframe.view([ref, model, moved], names=['crystal', 'AF2', 'AF2 moved'],
              superpose=['A'], style=['cartoon', 'cartoon', 'line'], sequence=['A', 'C'])

## Switching regions on and off
Click a region in the legend to switch it off or on; Shift-click shows only that one. `region_groups=` switches several together, and `hidden_regions=` starts some off. Other options: `orientations={'label': R}` adds buttons that turn to saved orientations, `surface=True` shows the molecular surface, ligands are drawn when the file is read with `hetero=True`, and `colors=` / `region_colors=` / `style=` change the look.

In [ ]:
ref.view(regions={'pocket': pocket, 'alpha1 helix': ('A', 50, 85), 'alpha2 helix': ('A', 138, 175)},
         region_groups={'helices': ['alpha1 helix', 'alpha2 helix']}, hidden_regions=['helices'],
         sequence=['A'])

## Arrows and labels
`arrows=` draws arrows in the structure's coordinates: vectors, axes, or pointers like these. Each arrow is a `(start, end)` pair or a dict with `start`, `end` and optionally `color`, `label` (text at the tip), `radius` (Å) and `group`. Each group gets a checkbox that shows and hides its arrows. Here every chain gets an arrow from its centre out of the complex, with the chain's name at the tip, in the chain's own color.

In [ ]:
import numpy as np
from pdbframe.structure.display import COLORS   # the chain palette view() uses, in sorted chain order

names = {'A': 'MHC heavy chain', 'C': 'peptide', 'D': 'TCR alpha', 'E': 'TCR beta'}
center = ref.coords.mean(axis=0)
arrows = []
for i, chain in enumerate(sorted(ref.chains)):
    middle = ref.select(chains=chain).coords.mean(axis=0)         # the chain's centre
    out = (middle - center) * [1, 1, 0]       # away from the complex, flat in x-y: the view
    out /= np.linalg.norm(out)                # starts looking down z, so the labels spread out
    length = 30 if i % 2 else 50              # alternate, so neighbouring labels do not overlap
    arrows.append({'start': middle, 'end': middle + length * out, 'color': COLORS[i % len(COLORS)],
                   'label': f'{chain}: {names[chain]}', 'group': 'Chain labels'})
ref.view(arrows=arrows)

# `py3Dmol()`: a plain overlay
Each structure gets one color, in the order passed (here: grey = reference, orange = model), or `color_by='chain'` / `'bfactor'`. The result is a py3Dmol view, so you can keep calling its methods, e.g. `.setBackgroundColor('black')`.

In [ ]:
ref.py3Dmol(model.superpose(ref, chains=['H']), colors=['lightgrey', 'orange'])

In [ ]:
model.py3Dmol(color_by='bfactor')   # pLDDT: blue confident, red low